In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [2]:
# -----------------------------------------------------------------------------
# 1. Загрузка сырых строк (без авто‑парсинга чисел)
# -----------------------------------------------------------------------------
train_raw = pd.read_csv('train.csv', sep=';', dtype=str, keep_default_na=False)
test_raw  = pd.read_csv('test.csv', sep=';', dtype=str, keep_default_na=False)

In [3]:
# -----------------------------------------------------------------------------
# 2. Универсальный парсер чисел (запятая = тысяч/десятичная, пробелы и т.д.)
# -----------------------------------------------------------------------------
def safe_parse_float(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "" or x.lower() in ("nan", "none", "null"):
        return np.nan

    x = x.replace(" ", "")
    x = x.replace(",", ".")

    try:
        return float(x)
    except ValueError:
        return np.nan

In [4]:
# -----------------------------------------------------------------------------
# 3. Определение типов колонок (категориальные/числовые)
# -----------------------------------------------------------------------------
IGNORE_COLS = ['id', 'dt', 'target', 'w']
all_cols = [c for c in train_raw.columns if c not in IGNORE_COLS]

known_cat = [
    'gender',
    'adminarea',
    'city_smart_name',
    'addrref',
    'dp_ewb_last_employment_position',
    'dp_ewb_last_organization',
    'incomeValueCategory',
    'nonresident_flag',
    'client_active_flag',
    'accountsalary_out_flag',
    'blacklist_flag',
    'vert_has_app_ru_tinkoff_investing',
    'vert_has_app_ru_vtb_invest',
    'vert_has_app_ru_cian_main',
    'vert_has_app_ru_raiffeisennews'
]

cat_cols = [c for c in known_cat if c in train_raw.columns]

num_cols = [c for c in all_cols if c not in cat_cols]

In [ ]:
# -----------------------------------------------------------------------------
# 4. Заполнение пропусков: медианы для числовых, 'MISSING' для категориальных
# -----------------------------------------------------------------------------
train_num = pd.DataFrame(index=train_raw.index)
test_num  = pd.DataFrame(index=test_raw.index)

for col in num_cols:
    train_num[col] = train_raw[col].apply(safe_parse_float)
    test_num[col]  = test_raw[col].apply(safe_parse_float)

medians = {}
for col in num_cols:
    m = train_num[col].median()
    medians[col] = m
    train_num[col].fillna(m, inplace=True)
    test_num[col].fillna(m, inplace=True)

train_cat = pd.DataFrame(index=train_raw.index)
test_cat  = pd.DataFrame(index=test_raw.index)
for col in cat_cols:
    train_cat[col] = train_raw[col].copy()
    test_cat[col]  = test_raw[col].copy()
    # Замена пропусков строкой 'MISSING' (LightGBM обработает как новую категорию)
    train_cat[col] = train_cat[col].replace('', 'MISSING').fillna('MISSING')
    test_cat[col]  = test_cat[col].replace('', 'MISSING').fillna('MISSING')

In [6]:
# -----------------------------------------------------------------------------
# 5. Временные признаки из dt
# -----------------------------------------------------------------------------
def add_date_features(df, dt_series):
    dt = pd.to_datetime(dt_series, errors="coerce")
    df['dt_year']   = dt.dt.year
    df['dt_month']  = dt.dt.month
    df['dt_quarter'] = dt.dt.quarter
    df['dt_dayofweek'] = dt.dt.dayofweek
    df['dt_is_month_start'] = dt.dt.is_month_start.astype(int)
    df['dt_is_month_end']   = dt.dt.is_month_end.astype(int)
    df["dt_dayofyear"] = dt.dt.dayofyear
    df["dt_week"] = dt.dt.isocalendar().week.astype(int)
    df["month_sin"] = np.sin(2*np.pi*df["dt_month"]/12)
    df["month_cos"] = np.cos(2*np.pi*df["dt_month"]/12)
    df["week_sin"] = np.sin(2*np.pi*df["dt_week"]/52)
    df["week_cos"] = np.cos(2*np.pi*df["dt_week"]/52)
    return df

train_num = add_date_features(train_num, train_raw['dt'])
test_num  = add_date_features(test_num,  test_raw['dt'])

In [7]:
# -----------------------------------------------------------------------------
# 6. Сборка финальных матриц
# -----------------------------------------------------------------------------
X_train = pd.concat([train_num, train_cat], axis=1)
X_test  = pd.concat([test_num,  test_cat],  axis=1)

y_train = train_raw['target'].apply(safe_parse_float).astype(float)
print(y_train.describe())
w_train = train_raw['w'].apply(safe_parse_float).astype(float)

# Убедимся, что категориальные признаки в X_train имеют тип object
for c in cat_cols:
    # объединяем train и test, чтобы категории совпадали
    all_values = pd.concat([X_train[c], X_test[c]], axis=0).astype(str)

    categories = pd.Categorical(all_values).categories

    X_train[c] = pd.Categorical(X_train[c].astype(str), categories=categories)
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=categories)
print(X_train.dtypes.value_counts())

count    7.678600e+04
mean     9.264824e+04
std      1.124090e+05
min      2.000000e+04
25%      3.970997e+04
50%      6.275413e+04
75%      1.002017e+05
max      1.500000e+06
Name: target, dtype: float64
float64     209
category      7
int32         5
int64         3
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64


In [ ]:
# -----------------------------------------------------------------------------
# 7. Проверка утечек (корреляция числовых признаков с target)
# -----------------------------------------------------------------------------
print("Проверка утечек (топ-10 корреляций по модулю):")
corr = X_train[num_cols].apply(lambda s: s.corr(y_train), axis=0).abs().sort_values(ascending=False)
print(corr.head(10))

Проверка утечек (топ-10 корреляций по модулю):
first_salary_income                         0.928217
salary_6to12m_avg                           0.927699
dp_payoutincomedata_payout_avg_6_month      0.672173
dp_payoutincomedata_payout_avg_3_month      0.644840
dp_payoutincomedata_payout_sum_3_month      0.644542
turn_cur_db_avg_act_v2                      0.640399
turn_cur_cr_avg_act_v2                      0.638592
dp_payoutincomedata_payout_avg_prev_year    0.630530
turn_cur_cr_avg_v2                          0.630285
turn_cur_cr_sum_v2                          0.630285
dtype: float64


In [17]:
# -----------------------------------------------------------------------------
# 8. Параметры LightGBM
# -----------------------------------------------------------------------------
params = {
    'objective': 'mae',
    'metric': 'mae',
    'learning_rate': 0.02,
    'num_leaves': 64,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'lambda_l1': 1.0,
    'lambda_l2': 5.0,
    'verbose': -1,
    'seed': 42,
    'num_threads': -1
}

In [12]:
# -----------------------------------------------------------------------------
# 9. Кросс-валидация по времени (TimeSeriesSplit)
# -----------------------------------------------------------------------------
# Сортируем данные по дате, чтобы избежать утечек будущего в прошлое
train_raw['_dt_parsed'] = pd.to_datetime(train_raw['dt'])
sort_idx = train_raw['_dt_parsed'].sort_values().index
X_sorted = X_train.loc[sort_idx].reset_index(drop=True)
y_sorted = y_train.loc[sort_idx].reset_index(drop=True)
w_sorted = w_train.loc[sort_idx].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=4)
cv_scores = []
best_iters = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_sorted)):
    X_tr, X_val = X_sorted.iloc[train_idx], X_sorted.iloc[val_idx]
    y_tr, y_val = y_sorted.iloc[train_idx], y_sorted.iloc[val_idx]
    w_tr, w_val = w_sorted.iloc[train_idx], w_sorted.iloc[val_idx]

    print(X_train.dtypes.value_counts())

    print("OBJECT:")
    print(X_train.select_dtypes(include="object").columns.tolist())

    print("CATEGORY:")
    print(X_train.select_dtypes(include="category").columns.tolist())

    dtrain = lgb.Dataset(X_tr, label=y_tr, weight=w_tr,
                         categorical_feature=cat_cols)
    dval   = lgb.Dataset(X_val, label=y_val, weight=w_val,
                         categorical_feature=cat_cols)

    model = lgb.train(
        params,
        dtrain,
        valid_sets=[dval],
        num_boost_round=4000,
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    pred = model.predict(X_val)
    wmae = mean_absolute_error(y_val, pred, sample_weight=w_val)
    cv_scores.append(wmae)
    best_iters.append(model.best_iteration)
    print(f"Fold {fold+1}: WMAE = {wmae:.2f}, best_iter = {model.best_iteration}")

print(f"\nСредний WMAE по кросс-валидации: {np.mean(cv_scores):.2f}")
print(f"Среднее best_iteration: {np.mean(best_iters):.0f}")

# Финальное число итераций – среднее по фолдам
final_num_round = max(best_iters)

float64     209
category      7
int32         5
int64         3
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64
OBJECT:
[]
CATEGORY:
['gender', 'adminarea', 'city_smart_name', 'addrref', 'dp_ewb_last_employment_position', 'dp_ewb_last_organization', 'incomeValueCategory', 'nonresident_flag', 'client_active_flag', 'accountsalary_out_flag', 'blacklist_flag', 'vert_has_app_ru_tinkoff_investing', 'vert_has_app_ru_vtb_invest', 'vert_has_app_ru_cian_main', 'vert_has_app_ru_raiffeisennews']
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1243]	valid_0's l1: 66937.5
Fold 1: WMAE = 66937.52, best_iter = 1243
float64     209
category      7
int32         5
int64         3
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64
OBJECT:
[]
CATEGORY

In [13]:
# -----------------------------------------------------------------------------
# 10. Обучение финальной модели на всех данных
# -----------------------------------------------------------------------------
full_train = lgb.Dataset(X_train, label=y_train, weight=w_train,
                         categorical_feature=cat_cols)
final_model = lgb.train(params, full_train, num_boost_round=final_num_round)

# Важность признаков
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print("\nТоп-20 признаков:")
print(importance.head(20))


Топ-20 признаков:
                                               feature     importance
0                               turn_cur_cr_avg_act_v2  476968.215878
1                                    salary_6to12m_avg  244791.407786
221                    dp_ewb_last_employment_position  182252.690145
218                                          adminarea  166694.959278
219                                    city_smart_name  153953.972692
2                              hdb_bki_total_max_limit  116859.772938
5                                          incomeValue  110766.154305
15                              turn_cur_db_avg_act_v2   80553.828208
204                                first_salary_income   68495.476376
4                           hdb_bki_total_cc_max_limit   68084.696430
3                           dp_ils_paymentssum_avg_12m   65246.414310
9                          hdb_bki_total_pil_max_limit   58196.846010
7                                   turn_cur_cr_avg_v2   56308.767452
3

In [14]:
# -----------------------------------------------------------------------------
# 11. Предсказание и сохранение
# -----------------------------------------------------------------------------
test_pred = final_model.predict(X_test)
submission = pd.DataFrame({
    'id': test_raw['id'].astype(int),
    'predict': test_pred
})
submission.to_csv('submission1.csv', index=False)
print("\nРезультат сохранён в submission.csv")


Результат сохранён в submission.csv
